# 📋 MNT02 미션 04 — 시계열 머신러닝(Prophet/LSTM) 기반 결전 대비 야전 부품 수요 예측 모델 고도화

**Solution 교관 전용** / 학생 공개 첨부 금지 / 교육용 가상 자료

---

## 시작 전 꼭 읽으세요

이번 활동은 **교육용 가상 부품 데이터**를 분석하는 실습입니다. 코딩이 처음이어도 안내와 TODO를 따라 한 단계씩 진행하세요.

1. 아래 **준비 셀**을 실행합니다. 필요한 라이브러리는 실습 환경에 설치되어 있어야 합니다.
2. 미션의 입력 자료·핵심 함수를 확인합니다. 학생용 `TODO` 부분을 직접 작성하거나 허용된 AI에 도움을 요청합니다.
3. 코드를 실행하고 실제 출력·오류를 기록합니다. 실행이 실패했다면 실패 내용을 그대로 남기고 수정 후 다시 실행하세요.
4. AI에게 보낸 **실제 질문과 받은 답변 원문**을 작성 기록에 붙여넣고, 코드·출력과 비교한 수정 이유를 작성합니다.
5. 마지막 기록 셀을 실행해서 JSON과 검토용 ZIP을 만들고 내려받습니다. 파일 생성만으로 학습 PASS나 사람의 승인이 되지 않습니다.

**인터넷/폐쇄망:** Colab과 외부 AI는 접속이 허용된 환경에서만 사용합니다.

폐쇄망에서는 사전 설치된 로컬 Jupyter 등 허용된 환경이 필요하며, 외부 AI 사용이 불가능하면 '미실시'로 기록하세요.

**보안:** 실제 부대·조달·업체·정비 자료를 입력하거나 AI로 실제 발주를 승인하지 마세요.

**실습 전 필수:** ① 환경 설정 셀만 실행하면 됩니다.

실습자료(미션 lesson·forms, 공통 안내)가 노트북에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.

준비 완료 문구를 확인한 뒤 다음 셀로 진행하세요.


In [ ]:
# @title ① 실습자료 자동 준비 및 환경 설정 (ZIP 업로드 불필요) (실행 버튼을 누르세요) { display-mode: "form" }
# 이 셀은 코드 숨김 상태로 배포됩니다. 소스코드를 열지 않아도 실행할 수 있습니다.
from pathlib import Path
import os, sys, json, re, math, time, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

current_dir = Path.cwd().resolve()
notebook_directory = globals().get('notebook_directory', current_dir)
workspace_name = 'MNT02_실습프로젝트_작업공간'
WORKSPACE = next(
    (folder for folder in (*reversed(current_dir.parents), current_dir) if folder.name == workspace_name),
    current_dir / workspace_name,
)
MNT02_PROJECT_READY = False

def _prepare_project_zip(raw_bytes):
    global MNT02_PROJECT_READY, MNT02_PROJECT_DIR
    import io
    if not zipfile.is_zipfile(io.BytesIO(raw_bytes)):
        raise ValueError('유효한 ZIP 파일이 아닙니다. 압축파일 내용을 확인하세요.')
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw_bytes)) as archive:
        for member in archive.infolist():
            name = member.filename.replace('\\', '/')
            target = (WORKSPACE / name).resolve()
            if not target.is_relative_to(WORKSPACE):
                raise ValueError('안전하지 않은 압축 경로가 발견되어 중단했습니다: ' + name)
            # 심볼릭 링크는 작업공간 외부 접근 가능성을 고려해 거부합니다.
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('심볼릭 링크가 포함된 ZIP은 허용하지 않습니다: ' + name)
        members = [m for m in archive.infolist() if not m.is_dir()]
        if not members:
            raise ValueError('ZIP 안에 실습자료 파일이 없습니다.')
        if len(members) > 15000 or sum(m.file_size for m in members) > 800*1024*1024:
            raise ValueError('ZIP이 너무 크거나 파일 개수가 너무 많습니다.')
        for member in archive.infolist():
            target = (WORKSPACE / member.filename.replace('\\', '/')).resolve()
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            if target.is_file():
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open('xb') as destination:
                import shutil
                shutil.copyfileobj(source, destination)
    os.chdir(WORKSPACE)
    MNT02_PROJECT_DIR = WORKSPACE
    MNT02_PROJECT_READY = True
    print('✅ 실습자료 준비 완료 (기존 파일은 보존):', WORKSPACE)
    print('✅ 이제 다음 실습 코드셀을 실행할 수 있습니다.')

# 실습자료가 노트북 안에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.
# 내장 파일 4개: MNT02_Colab_실행안내.md, MNT02_M04_forms.md, MNT02_M04_lesson.md, MNT02_공통_작성안내.md
import base64 as _embedded_b64
_EMBEDDED_ZIP_B64 = (
    'UEsDBBQAAAgIAAAAIVz+D1bE0QYAANcMAAAbAAAATU5UMDJfQ29sYWJf7Iuk7ZaJ7JWI64K0Lm1kbVdNTxtXFN3zK55UdVkH8lW6RCmqLAFFcbLpBpngSG7B'
    'tmyzSFcGBuRgU0ywwziMydCYzzrN4A88qGbjnzPvzX/oue8920PCAhkzb+4799xzz718x2bnXow/ZM+Sy9FFJgp1//1bJip5vt4eG5sIqacLmezqUiyRDcVT'
    'bxKL/KTHxEHba9lM1HP8xmBiMycOSsKw9HGEEdsdv2zwY0vYNX/bDf0ZT4kaTpoNfnrLC3leqIeYsA3RbAvbEkd7vOWy38LzXqvHvE6DXxt+5WJwznMdcdxm'
    'Im8Ku8L83TbfKfPtMhNHDYXYr7T5rsH4zr/ifVvsO+IshyS2AUKHGHsYYn7FEB/KfsVivNYTtTZ3i/wj4DT4WZEJoy5uzMCdidWV1Jt+NxVNLEUz/W7mVZy+'
    '4uOPePaH5Vg0neh3V6LZ1HIyuxxflLmtN8SHi0AIUXX5dY5IRYoA5+BS33D63an5MPPX6/2uODqR/B04XpOeDjGa32bwKMT4F1dsmmwmlskkcf3rZHolI2om'
    'EeabBvNahqjcSii1WyoPOBU1l04QNttgfrHEC+cAEEa9GD879lwXh3vyHbsSxK5AT4UZ8Ijm536XFzq8lQPm5j9MtPPQSL/rdXvelcN4MSeOkRdvtUXVwJGO'
    'Jf7eBKe2v1Gj2PTguD0KP/Z4kAxvGVoGbCn2mqQVX0ktx1agtmg2nkww3nDFUQl6I8TQiTCuUG2WjqWWo69iC69XE6/onLB6EBtC5cBpMA/U+ZQg1SgMELsO'
    'WPGcd6IGgi0bPN8jtNsy37dIYaHFGFiOUWiVAy+ZUvhHJr/8j5cswsebFVxgWPxyk6KCEZSdl4ZR55LZ8CCp2NJ0Op1MM7Gx5m9YqmuklKWOSQKivocajV4f'
    'e0LS7YhCAx/n6IFBuf1Dg6VXEwsr8UyGGKAaNk1RK7JhUzDICh1FtH5dAmKGIgxSvi57Tg6ncl7zFkrqofqSUEkF0gNPe/jEU8gsGPfG8DqlQFzUVQMkGMVz'
    'aRfGFd8+kXfVc0S+rtY9naq7XLapUsh1Dk0T7PUhM09DLLmaTa1mMw9meTPvm+4D/0NROOeklUx2CQ/RstmlWDpNyAmySqLr8tNevwv2Qr9niDoQAKUC5ahq'
    'fHcTVsX45Tnfz8u6OjlUDUBK/FORsvKcNXzRb6Jn+ZeczKpypKHLzpME6qzkRXh/t6oZkNHtwXFSuL0GE4HKQIzUxR1uvFbH3+roKsmCHzTgYQOQw/cJzEV7'
    'UE3eLAsLPcCbBrVaTRadH9cCTP4Ip/+RibM1Ye8B4Sa3TwKuPsjcrqDa6C8UHDd/9is2cnjkXbVJTHTCa+b8LTsoMviOZUM+Zl6swyFQIr9qgi6mrINo9Kvg'
    'xQVkaqyZ6annc9PPF8Jz8y9f4OISImi49xkJo8AFi0XCsy9npl5M/0zKhFSk6FQ2yvOG/mCJa+s+fx04nnJxWVdYxmkNtDmW+ETh9mSH4r2DrYAvT4K6SVT6'
    '0fj491Qxf/ctENEfuOMQlXeGnbTZagXTSlQhKdVNhKrqjh5JcekhKFmBPnihQXer3yiVr2vIqFEqEJx7LJVGzdyhjPx3eX55IecEBgRVAanQr7DGC2eY9+EN'
    'KlEm2XjOlSjYw0DAXDF4oTpI+CckPDFO8Uh2O672eSo++JIXq9ZRYwB8QD9kLnfCYK60hbkmtQUg+9bEuOcAxtGJ7Bo9M1su/BWAMSDJdbqW17yQcwaskHNS'
    'mfVMGEni2oJ80UJUDcrcOVfLAp3DUDbQvt/CoTlDUu80MMpkLutSReg3sg29YKy3IQ5qMtyHaIplgx4p6Y/qPTYxLueJnMG5iXE9uwcqyJskpoA6Dku0AWkq'
    'Zx/3u7NP8TOpJjPJQ/d0ZHpm+hmEvjAbjkTCv85FSEcUXEbUDScFpPUhBQSnF1s72rugbBNdJPEbw/Yb8vdsNiL3kLz5QMPRpaIdaaAsmbN6NZgzNkZ+lifv'
    'OnsrDQSH9fBSQ4EGJcWUo1NTopZOhZfEKdtW8UnmAr+WNgDRw7gpXb9YxNLCKAfV6ncsZBU7WXQJI4F20cHiiR2P738OzL5BidXs1bsOFhn0QEBgENMFHiNZ'
    'eFelSlutlDObn4pEAKxT8tpQ0vaN3LMM7Rn3+4uepMPYtIbJujfkq8F1We+4cknTjge63tVJgNyuc+cwYEBjSjtgbiPPfolnmf+Xq7X5cDL0BGbkOUZAaqiv'
    'bLGtIi0T6rFenOQ8uZaDh+9foRxyyfhGIQPLQAQshyr+SBXKN4KCkv9b8I+Ov4FLDNoeUWwQLbtBDi21tanVSgXX/5CgxzFytrErS1lqgxo5TdmRtn5m3N0M'
    '/gdQSwMEFAAAAAgAAAAhXOhbBbHZBwAA5hIAABIAAABNTlQwMl9NMDRfZm9ybXMubWTVWFtrG0cUfvevGAiFFhIrdS8PfikpvUJTCs1bKUY0SmIa20FWCwU/'
    'yPEqKJaKlVRrr+yVukqc2E4VspZlaw1KH/xT0red2f/Q75yZWUlu4lJoS0tMLM31XL7vO2d8Tlz+/MrFKXH54ttCte4pZ1+o9Xuq0hQvinWhKn584KiNnpC7'
    'kaoE8ueOrDRf/yK/cOtGrpD57Msrl98QcRTK0BNxN1SBI2S1KI8dodwBfzsqJvfLQpU9tYnjvLKK2kI+2ZU/RiI+COSakzTqExNqqyYPIpGs3Y3D4rT4SjlB'
    'stL8+uX72LKkWlUrTZHUB6rriTcvxqEvVLOnVpbPj1gdH9ZUuBt3i+pxk0/vB+LypS8+FFPvvEbLE9c7b89Q2/dwFywScX8Q74eqUTztk7HmcRNWCvgln+yp'
    'JsYDH1cm62W5WsfM5MSEPCjG3efwEEHwp8Vc9t2TfiGfm786Uzjp38xen+I9jQi/nGTLGbOTTsRtGIJ1iJH1HleooKY2aohtVQbbagf7Dpxk80ec4sqDHhms'
    'Ktu8lgfkowFO7sgjJ3F9WdmGYefOGXdP+qN3TkwspeOy65DPS+Ze+qyPM3HBTNJwVeuRCp+f9DmaS8PgnfS1ZzAk2fAwI8sO4obr2FSEB4eJJdx34cIF8cr/'
    'MS/Gfv67AxTTS58SC+Af/Ax9Gfr4rUFH5GgjvheEut2hWCbVmqzsplzjpE6LF+ttOkRWDgEdgWBjH4GDJ8woAC1bVUF40btx6KVPCRBypx1HANNGSXY0DMAE'
    'P5imWwNw4TBxAaNWTT6sIj/P6/InX+fJfQ6ICrAPUAN4Ce8YoH2go2rip8cwWFkGgswsn+86ameZCfmoRxvhXOI+nSYcgSzgtCBuBMsEDUAyADk366r7lO6S'
    'nUhIr8bbgiKAJJTvqGNvuNY6fOZaa6En5F4dlpwFqokzcgj1YdOAfL5WNYupzWwAExL0akMu9mSlTEwSOorCEN2YaHPOcSL2QReVx7FjfjomCiwyDIzhiRQ6'
    'IEA52/JJKfVZ36LlkOJjcwh2rd896SelqlrdTlajITeHYdFI4wQzHGi/H6idIvZDyQIXHpNIcOAPffWgZOxlcW0OsMzE+p9g68Rb8X7PSDZHuFpU7XA0wuyr'
    'kN1y4kUUTBX+qoO8QpiP+xFsgzdAf41kMnF7snIs5O2e2oQs9+Rtz+aRqsHt0IZtJD9AqFo9HMmAzieDHflZfYQQV6uyOZDPosTR4eWEsyoaIqWCaeaYoKfn'
    'jNCPWTCCex1+k2ebrr8h7CROprjCIbVd5GvNbRCAI5+ln+suIvgUrMVGszS8DweOHHALchaFcegMKao9wHiKu0oHkoQNzyJ8pLjVSOnlUZmOXrIE+HeUn4QP'
    'TvbKaSLMV2JYr6zcsmG0SQErpU1ZKQKh5NEy0wPIUg7JeT+Sj+GvbPv4aGI3shsAUk2g9VmkSp7yB1gnpOORLOuaRxXBiitKb9yj7qKY3IEwQ11XfF0DwAJS'
    'fv44qhg8olaPQe3finVZIaiDIPCrOUANsXn4rejqGu8eqkqHin/iOWkNQqLdXVCfhaQkA6AbTuA3F/RmROPdX4QVFy0SYgS2Q+0N/bjbIZoobxsKki62cDhT'
    'iuey3+ZmruWyhe/yuUWM8z80Tc8iYzBdV91lAXT2wcIhILkguY6sNFKZIIudfRQ5YyUj/IgasTFC8txobzUUXgpYw5drDXmAhTsOe3+qfi+ZJab7aVo+jTCC'
    'XR+Kpskro+hPkI+ZN/GZenDdJk2/ovM2wPqrrTa5nXbbuhcc6+FNyaMUJw1P6xs1B+a6ZK1DBf9Om0RXtjxZr6t1IgndSfWnWrKIh4h3e29MkkAn2LJZZ4wY'
    'sGkNoTq7iyAhUuiINveIH5Rj3QpR9sbwgUqG/QanFBKYRT0310nTIac1T/dYuvCd9NEcIRctxyZTx5aDoYuv7djfMxBcElM2DQzEabGY/yZjLcnMzRcuTlnD'
    'ZnLz12fnc7n87Pz1yVs/UMEdsxvlRXcs2kZtObh7Y3axID7IFrIf5bNzOfHizn3zkiDwwXuij84hN1rcvsmab1t9alDKHsd6pyp/Ck49hNKDJwWeGRkqg2sN'
    '/fDJkAlBGRWTDgC3SKlbnrlMF9iISzBXVeWuYhFTjLRmC5Da5MoKYIYOI0Y3tLaxJ25SpaQSzYrHZHtSoqeRfn5AnTCmjvZUe980nGCvJq8VHa2vKB/3/IzW'
    'XGwx3dcwXfR023I+yy0uLsxT2pnumu10AxNS6zmQaJqwPyb7LZtsRsf//tlphChlmancSCk6La3Qx4QcXZnt45LrYkbLdyqSRiOHIRtXC5tn01ngUmsS95eU'
    '29qw+WIqMWB4OXk1JiqfgBEf57NXZ3PzhfcXFhYLYBRJhhX4rTpRgzHUASxbIr9w8ybWmLaFAvYu5wEYpXJNZQjaHNTsC5YIb57BaXd5JqZf9sh+KcB7qcrp'
    'YNrniz6bvi3Tpr2eeQaPuEWdyMMB9CGjUUftMCzVICAWcHNgqwfS1+ro9LwEyG8bIGMhWua/oFrnRT53ayFfOL3q++zN2avZwuzC/OTc1fPi2kJ+bjHDfyia'
    'wT0z/B0zRD3NNX6P6i7rQUk3jvwM1MzVJRHT7KlubymEGyGQDqhYf2zvSF0U/eWBXih4fzk+Ddgr9IXcCuluiZXWRVVOWmXIxignbIh+B1BLAwQUAAAACAAA'
    'ACFcy5eQFX8JAADoGQAAEwAAAE1OVDAyX00wNF9sZXNzb24ubWTtWF1vG8cVfeevGMAo0ACqaMlJWuglcBs3DRAVRpOnFoXEWowtRB8GyRboGyWtBFqka8rm'
    'SqS0ZJYNZUsJi64oSqRQ6oU/JX7bmf0PPffO7HIpy0EENA9BAwRwuJqdmXvvueecu7fE7O8/uz0tZm+/K77NV4QqOv6ppfa6Qh71VdGVX7Zlsf7z+5nVx4/S'
    'ueQnn342+47w+570qsLveMq1hCzl5YUllD3gX+f54HlBqEJV7WO7akH1m0J+fSSf9oV/6spnVlCrJBK3bgm1sRbUjv3TAf58HOw4iYTv5fEw3EI+9VS9G1ge'
    'HljKqgtZ7MrihWqUZdNRlmPOGPZk8Uj9c5P323CGPdVo4yDh9wb+iScPB+b1wD6WxYIstibfEqRyBtj4xlGpujUKLLAdnDCWLlXP48BWsPtEBLWq37kUynWU'
    'fWmOC561lW2prSYu3JeNqqxU1G4/2O7TmWq7FZQ2zUrcSHW670zq5B2U5WmfLkQby05FOZaq5bHdE2QxkfhFtMJygw3k7t99tVlF6K6FTehCavtM7ZWRR7ld'
    'MfuMUhS9j3RuFMRHizkR/KOvrBNxh5LnWa/zFdqj3/S956gS8mTJYvV13o6dzHvOCPUqr/Y8rMNe9a58VqM79lzU7TSPX9h02KNyuAVaWagOe8FBQbpHyKoV'
    '7D/FW/tdtX+szxXyxUmYfzxRbllgUyqRcm152uWUlkoKIQeVgepUxdRt33N40cYa1Uq+KskXLuMgqNmqcSiUdxmDhH9WVt6R38mrlyEkZu/ev0dAovX1fmBX'
    'CV+q4NBy1doBJFB9WogXEbxQ57baKongwKIb4UXCx2FfY6+xQ7cNNh1UXkxNvic9j06X61X5sq2qLV0UgL2l6iX8045Q97KO0uI/Eax5AnBE21D3ADfAfl2t'
    'e6YF4ld1TDX/pHHw5+sbk1v/7WmbeEt6hK4k50dMv/czWo7sTIR7RLkxzUgAvdJe8dBMTFQkapGiE+wWAE78ZTKGqnVPnTuyPZgR2cyD5OfpVO6vmXQ2ubyS'
    'uz09Z37OpVceLq6k05nFlYeTj/8upPeMSEu18iZi3UMugjoTRChflahl8CvYOuMSAenKLsj17rD3m9Wl1F/CFg4f6gYScpNaVZ6vhU+CCgFBuXXq4D9+fJ9v'
    'flmRL5wZMf+9LzwPbNpnCrUP7CPkiDF01g6qFs5x8wBtCMLONxG5HFji7seUYnVOORNoL7DNWE9HlDojxi46v5DKpZJ/uHf3w9l7k8sL88yaR2A+bFa/RFhm'
    'izu/DNDnL4CL9TaqOOxN/Sp8cFBGwUwuh73pKUKPQZdqbEr3kHnyrA0yHqcZNHFjR6jdHVWsI0csR3OQo7nPVzPLWboNl29h9UE2mUk/Xs3ksqpeZUKrWVwn'
    'Xa6gCJjouoJlOgVDebTW5FJnkApNUK8PblQReotrgbeWU1+kw3XZeQaOBUafEY8WsznxIXL520xqOS2+3XpuMC1PSSSoBwzkuU6MP1l2DNNRxajYxPVMUlda'
    'Mtp4UiylHibjZJqMEygROoKHmJjDqJzK66umB9hgDVC8jUXMR6SoB+jAGMeyYnGBzcU0x8v6gHYmkXZtZtKvN6lJI46BJ5Dop5FuEytRZTSGSWZMx3FDECyM'
    'IM9OUZkePEo/+GK0zED8b6mlRcAzPeqBMfyM1ZabgHvCOuGDuDX4rqdd1eyaLYkKmt3xPqHy4FxOlRHDLqkl7QimAu2NGYgW1qLdiMcg+6RlMAJ9w3MguXyw'
    'BYUDnctDXQXXZjJ1XO6t9SP/Yi0MgjAb2DXik6DmoKb05Jb5/ynmZlMHsL0pTFAqw/YkEoYVtYOCjsKZEUlygskKUTtyTPVLipyLBcPR0PyPrEcVxuYhaCg6'
    '1kq125W7//E70CHIU6HkX5Q+YHXh02MJAfHslYHapt/vGzcSnk6MRX4Hss6aBJgcy46tWd6O1TIW9DQHbdiMYzFVSiTewotv8iGEhkKibJvaY5PnLbMrEoWw'
    'DSfhfbo+Bb1/HItqDBkzYjn1/rCXy6RXFuZywx5acJp1qtYPcTJmHeoj3OirxU0LH2eXpNtSr8IuMw6GY2ZrFVoaHSQTp3M1U3c0PDTsyCE2WWXhgXUagBAD'
    'Pk4jQ2FMQMKFsbBD9Tax4c97Ve1KfK9G5KB9E2wTAw6MAfyfW8T2tL3jgmIi6f6AukRZLZAFkKBrqb3IHf+kG/NlEVr4OjwWcDjmopwZpLR4xrbRYoMExQmN'
    'znUwMiPNmUPjgcasbiFjIaL20a7ZWAQ2ptqnU1XDW2uFv9r1XdyNPDTn1cG2cfignYptrIIQoddcO9pCbV9QJ4zs2lVhFOMTRMi9oQcCTxmfpJNtV2W5GrKS'
    'ATSROC8nqzU2aPwOKvVRJrWwmF7J/Xp1NZuDxMXIGHJAcqWpAihtiMzq0hLWmDQRvt5nc3hAxOQSuihzZMUNq7yp9N+tM9e1w7Wi08W1LChaiF4dcbi34a2u'
    'PMZkV7AI+VFYhmW7mASBWm1f1VcDMCAunkj8lO8fIt+6Jbjnaha623QYWbCbWa954/yuLDTeYHF1hXwi1rFnNI4QpZiIKTWZrs6/aAbdAYlMMC10C/gd58eJ'
    'UD1QQcxVQAqG8pcD0rCYE4iPI6DoL5+YuNjlsexQzTQljihS53SCXsHtkDlDSYa9aVPs7Xfz5IVCbLGRiBEFjjCzJkl9M/5VAxBhUuFMMyExy4TfGXZ4oor8'
    '38jUG9PMLMk2pmrok1GpNTus29QkfynSiJn5kX9IoS4I4e2YKSWmaMUj7WLQMfvH1BFy+9DINyF6bAggGmlskqq4h2YaYLpg3YsUjCwireQe1MUf9uBHHEs1'
    'rJCVdG45GbrCkZImpk32mXBuMP9SSceuC3TEXYC+8Ot85f94fnmdp2fq/Fg1T4jfif073+g5QqMOkDCkgF5KamLAK8ZdjvwODT8H1ifpbHZ1harNU4a2FHQC'
    'N7aZgPbKmJlUdS1W4zumxoyFmR/7dxvDY+MuET3V9GD+NPdeEGDQDDwMaJVhZk5qYo5GOGOmR3n+SbL/x5LNKglL1DhMjhw6bmq+pAP8ZqzVXpy/tJvhYYTf'
    'dzV+tQrdgKMmxPeQ+Akt8Mk3PxJRo+nOYu3H3VjP+Kuzduu6T4lrL6r4MweoJ3LK3J4HgAMhIbiu8e6y3SenjwfhEfpAVlmcQXRNvGpDM4NGwQjuKDOJ2Bf1'
    '7/igntQb6tpGn9UnzUcLPYsiWj0Hy3XCin5IRKXn/e1KCPpoKvovUEsDBBQAAAgIAAAAIVyxBwko6AYAAJ8NAAAcAAAATU5UMDJf6rO17Ya1X+yekeyEseyV'
    'iOuCtC5tZG1WXU8bRxR9968YKe+YkI+mjyihLVJAUT4UqS8RTWmCRHEERFX7tIY1MrZTTPDGa7LrLI2NgS7qYhtsFPPin+OZ/Q89d2bWXj4kPizvzp1zzz33'
    '3HuLzcw+H59gg9ZpuHHKRHVbmCdMWFm+1k4kBoEh1tNMZG2+7/MzQ2x8YPxDINx2aAZ8z6EnYrck7Kzo7vU7ouoPWh7jZ6Yw3X6H5w/EPxmGCOG6wwZNI9zw'
    '8FLuXLhdNugigCtck4l8TeROQ8vmuVIsugwS2hYLSyauEp4b5rqimuH5LM/Xxpj4XOStrnBtFn7M8qPDfmf22Wy/88Ozh7gDsM5tiQApUNzBaVE4e2L3kKmc'
    '4oFwv+cAtcELRr8zaJnCukCMckY025RUnZ8DZbXIvxZ4vcfoqFcHXtEwwFQO4HWoROLWLTY5zcKKg5skNMsEFrDbE56VSNweY4vzKyupJVzTDUQN5y9KfMeR'
    'Yd0LSR7lhJ82olj4FFpgec0HSEBxbe53+x1cIRxTVE3GmyV8QrSmIfZdxoND3rSIVH6AGxHeS4/gMZ4/5S0j9pi3a8Lp8UNctu2IvBNah1EqE2MsgpivhZ82'
    'Ub8AaRBS3mqLvTYIILyT06Jc1Oz0O6ENWL7OCoA9R3H5uRjVt2UMmhey7N7lG6M6kJ6axzjT/DcGGGhRiBi+O2NSfJ4VozAsFITbIzw8sIWdDq02kPkqgdhN'
    'xN/XHuLhFnmYCdML1112Z3CCE26bBOKaQB3lRaJYC5BfpiByNRJihdQi9oIYpLuAVLX50TdeVHKChKEBoqti6rwk+cMSq3bTTOp0YkUCIXlnqIMRUY4H6SmV'
    'exZU3mrzug1E4tShhlPJS4mc27JgaweD83QE817UOyDV4l82dZ0RTxVAFRph/+vyvC//o7vlTdQ78ka86RYYPcnYTJQDnCHEqqljUF++/TP5U+qP5Mu3c6so'
    'v8qcXgt3P4wuDBweOMmhQagswkKRDAQEQtmDk2DUseRLaV12SXDkGbAaF3wKF8reyvD9C61dKSDHpFYaNNfxLj8MNHgqc9YRpoNqkstoqbThh6AZ/VtB/ygH'
    'o+Qhh62K8EwQw4sk6MGpDxHgg7DwZWXYaVLmI2aVynjTxFnqVtlKuZIOXy2SS0lONapyUTTSwttW1MZrrysn9Tpzt9+ZuY/fB3yLIsOvyCrARbpNIZAVgGXt'
    'S+qXJqUQKIdqw2cTCRUYUNez7MeFVRb+3dVUTjwYuwcdDgLzmvsycn569JF4RALdPfoMS9lzOBhc82+ySdihJxs374AVpI4+yIpPbSTB0b4oT7fAv/gYOVIi'
    'ZD5hxSbbKBd5KcA7uIQeSUx4HcXAuUaWGgKRYoZ3dAhjY0rJyGXQ6ao6XxMqoqPHMSZ8xJTeVOkCI3qPUIFFyS4bfVXvUf/nvZvnAMiFZJPU+p6VVNVKhlub'
    'lCzuxh0kgDOTHx1QpKPMFSuEWMsUlyYv2kYNUrq57pE9kdayEDEGEwYN+rFgkNQkISzcPVDq0tNbNW2sdlppeV9brh7icDTdNLttMs8ozLD/b5yZGL4stLso'
    'EORCmipnuA+iGtuy8Qw13fWg4sGneBtDiGobwDnJ+ZkTCXHmO9n5st9IrbBxQHk2PfPi8eTzqUdaeHFEul+OQ8vTIxplQj2vtBnF8mhERmPceTw1+XR26umr'
    '6dknL55H2jXp8FWRUB2yGBbqEoxihBTlNglrtF7gxjNTeUw00dCTWAeqGJOw/X03quDILaCCHaA+w4RwKJrKVxXVhRd5k9OkQcUGzQPI8KbGgo1isA0tlLA0'
    'alIeEC9VCjKmSrmkCBJPtQ7kugg0vXUd9bIm+/f4ci6yrkPLN0R5I657vcvFNrafp5+A7IepxblfyJJsn3u18KOBnHimi+f8LK284/KIxsTE4A0/mxxt3TB4'
    'Y1NkDOVp798tpuZ+XUnKxXXsr4V3OM8bBb5zPDwMxJerrkYlih1tJlQ805HUhJVS5DPDHQcVkSSRJLnfi+lsYWlldfn969XUMhNBl5xMjxgDr9IBSAJrGtL/'
    '5g/HDxE4hINaji6lE5J9SiJmKNf3SorSRAVlaaPNV5o0dY+aNIlEnPqhhcccXMobmz72pDjbUKRHywq1BqiS7+rjtMJlEUStduqeawrWmWwUyEjkaeIxD2s+'
    'J0J0hobqq3jXeg4mLeaNxO3jr9qtNLVq586VVlLvl1/Pv0otL7xZWJpbXEm+WVh9tTz/2/zy/BK+nxifuH97fPx70scl/yey1rH4HDydmnw0M4XAKouF39+l'
    'lldp6O6coMV1Ulenm1retXtTgeXeRYts0b46bV3j9jjtMfXe1XnLoi+caOOw7+BNNQzQXphRVHtYwHbx6qqX+B9QSwECFAMUAAAICAAAACFc/g9WxNEGAADX'
    'DAAAGwAAAAAAAAAAAAAApAEAAAAATU5UMDJfQ29sYWJf7Iuk7ZaJ7JWI64K0Lm1kUEsBAhQDFAAAAAgAAAAhXOhbBbHZBwAA5hIAABIAAAAAAAAAAAAAAKQB'
    'CgcAAE1OVDAyX00wNF9mb3Jtcy5tZFBLAQIUAxQAAAAIAAAAIVzLl5AVfwkAAOgZAAATAAAAAAAAAAAAAACkARMPAABNTlQwMl9NMDRfbGVzc29uLm1kUEsB'
    'AhQDFAAACAgAAAAhXLEHCSjoBgAAnw0AABwAAAAAAAAAAAAAAKQBwxgAAE1OVDAyX+qzte2GtV/snpHshLHslYjrgrQubWRQSwUGAAAAAAQABAAUAQAA5R8A'
    'AAAA'
)
_EMBEDDED_ZIP_BYTES = _embedded_b64.b64decode(_EMBEDDED_ZIP_B64)
print('📦 노트북에 내장된 실습자료를 준비합니다. ZIP 업로드는 필요하지 않습니다.')
_prepare_project_zip(_EMBEDDED_ZIP_BYTES)


## 상황: 어떤 문제를 해결하나요?

**미션 개요**

- 시계열 머신러닝 기법을 활용하여 결전 대비 야전 부품 수요 예측 모델을 고도화하는 실습입니다.
- (단, Prophet/LSTM은 실행 환경 제약으로 인해 포아송 그래디언트 부스팅으로 대체하여 수행합니다.) 배경 및 필요성 전통적인 이동평균 방식의 한계를 극복하고, 전시 작전 템포 증가나 대규모 훈련 등 급격한 환경 변화에 유연하게 대응할 수 있는 고도화된 예측 모델이 필수적입니다.
- 예측 시점 이전의 정보만을 철저히 활용하여 데이터 누수(Data Leakage)를 방지하고, 정확한 수요 전망을 통해 결품 위험을 선제적으로 차단해야 합니다.

**핵심 과업**

- 파생 피처 10개 이상 생성: 지연값, 이동평균, 변동성, 계절지수, 훈련 계획, 운용시간 등 예측 시점 이전의 정보를 바탕으로 파생 피처를 구축합니다.
- 시계열 교차검증 및 차원 축소: 확장 창 시계열 교차검증을 수행하여 평균 MAPE를 확인하고, 순열 중요도를 활용하여 차원 축소 후보를 선정합니다.
- 작전 템포 시나리오 분석: 작전 템포 1.5배 시나리오를 가정하여 다음 달 수요 증가가 가장 큰 품목들을 식별하고 그 생산 상태를 확인합니다.

**검증 및 산출물**

**검증 기준:** 파생 피처 10개 이상 생성 완료, 시계열 교차검증 평균 MAPE 25% 이하 달성, 피처 중요도 근거 및 결전 대비 수요 증가 품목 제시.

**주요 산출물:**

src/features/mnt02_feature_engineering.py 및 기준 모델.

## 📁 이번 미션에서 완성할 것

- **미션에서 완성할 내용**
  - 지연값·이동평균·변동성·계절지수·훈련 계획·운용시간 등 예측 시점 이전 정보로 파생 피처 10개 이상을 만든다.
  - 확장 창 시계열 교차검증으로 MAPE를 확인하고 순열 중요도로 차원 축소 후보를 고른다.
  - 작전 템포 1.5배 시나리오에서 다음 달 수요 증가가 큰 품목과 그 생산 상태를 확인한다.
- **구현 계약**
  - 핵심 구현 계약 src/features/mnt02_feature_engineering.py / make_features: hist DataFrame → 품목별 과거 수요와 제공된 계획 변수로 만든 파생 피처 DataFrame. lag/이동평균/계절지수는 미래 수요를 참조하지 않는다.
  - 당월 운용시간은 사전 계획값이라는 가정을 명시한다.
- **변경 과제와 검증**
  - 추가 변경·검증 시계열 머신러닝으로 결전 대비 야전 부품 수요 예측 모델을 고도화한다(Prophet/LSTM은 실행 환경 제약으로 포아송 그래디언트 부스팅으로 대체). ma6·trend_t·lag2 제외 전후 교차검증을 비교하고 예측 시점에 알려진 계획 정보와 실측 정보를 구분한다.
  - 핵심구현 후첫성공실행을보존하고추가변경·재실행·실제AI원문·근거검토3건·보완·최종판단을 작성합니다.

**제출 경로:** `src/features/mnt02_feature_engineering.py`, `reports/mnt02_feature_validation.md`, `forms/MNT02_M04_forms.md`


## 🔍 교관 검토 기준

### 정량 평가와 측정 근거

- [정량적 평가] [선택] 수요 예측 모델 — 파생 피처 10개 이상, 시계열 교차검증 평균 MAPE 25% 이하, 피처 중요도 근거와 결전 대비 수요 증가 품목을 제시했는가.
- PASS 증거 제공된조건·단위·기간·분모와실제수치/건수/항목근거를연결합니다.
- 목표미달·0기준·미측정은그대로표시합니다.
- 보완 방법 기준별빠진항목과미달원인을찾아수정하고재검증합니다.

### 정성 평가와 검증 근거

- [정성적 평가] AI 활용: 본인 계획·실제 대화·코드 변경·검토3건·보완·재실행·전후 비교·최종 선택.
- PASS 증거 Prophet/LSTM이라는 원본 제목을 유지하되 실제 구현은 원본대로 포아송 HistGradientBoosting입니다.
- 월별 한달 앞 rolling 평가와 6개 월 고정 기점 예측을 구분합니다.
- 당월 운용시간은 사전에 알려진 계획값이라는 가정이 필요하고 실제 당월 실적이면 누수입니다.
- 보완 방법 근거없는원인·규정·완료추정은수정하거나확인필요로보류합니다.

### 진행 추적 및 완료 기준

- [진행 추적 및 완료 기준] 정성·검증: Prophet/LSTM이라는 원본 제목을 유지하되 실제 구현은 원본대로 포아송 HistGradientBoosting입니다.
- 월별 한달 앞 rolling 평가와 6개 월 고정 기점 예측을 구분합니다.
- 당월 운용시간은 사전에 알려진 계획값이라는 가정이 필요하고 실제 당월 실적이면 누수입니다.

### 제출 파일 및 완료 기록

- 파일: src/features/mnt02_feature_engineering.py, reports/mnt02_feature_validation.md, forms/MNT02_M04_forms.md 실제AI요청·답변·근거대조3건이상·보완·재확인·본인최종선택과작성/실행기록이연결됩니다.
- 최종ZIP생성은실제학습완료·교관승인 인증이아닙니다.
- 보완 방법 미작성·미실행·검토누락을보완하고교관검토상태를별도로표시합니다.

> **주의:** 아래 구현 코드는 원본 교관용 Workbook의 참고 구현입니다. 실행 결과와 기준 달성 여부는 실제 환경에서 독립 검증해야 합니다. 승인·발주·실측 사실을 생성한 것이 아닙니다.


## 📋 순서대로 할 활동

### 활동 1: 계획과 사전 판단

- 원문 목표 및 시계열 데이터 분석 범위 설정 원본 목표 및 기본 제공 코드를 철저히 분석하여, 시계열 머신러닝 기법(실행 환경 제약에 따른 포아송 그래디언트 부스팅 대체 적용 포함)을 활용한 결전 대비 야전 부품 수요 예측 모델의 구축 방향을 기획합니다.
- 한 달 앞 예측 시점에서 실제로 가용 가능한 정보와 미래 실적 데이터를 엄격히 분리하여, 사전 정보 유출(Data Leakage)을 방지할 수 있는 피처 엔지니어링 기준을 수립합니다.
- AI 위임 범위 및 검증 계획 설정 make_features 함수의 입출력 계약(Contract)을 정의하고, 파생 피처 10개 이상 생성 과정에서 AI에 위임할 코드 작성 범위와 활용할 자료를 구체적으로 확정합니다.
- 교차검증 평균 MAPE를 25% 이하로 달성하는지 검증할 절차와, 피처 중요도 분석 및 수요 증가 품목 도출 계획을 사전에 준비합니다.

#### 확인할 사항:

- 이 활동의 확인 근거 M04 계획: 시계열 머신러닝으로 결전 대비 야전 부품 수요 예측 모델을 고도화한다(Prophet/LSTM은 실행 환경 제약으로 포아송 그래디언트 부스팅으로 대체).에 필요한 입력·기간·단위·사용 가능 시점과 make_features의입출력 계약을 설명하고 본인의사전판단·AI위임·검증계획을 기록했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 2: AI와 코드 변경

- **함수 구현과 첫 실행**
  - 핵심 함수(make_features) 구현 및 첫 실행 보존 품목별 과거 수요와 제공된 계획 변수를 활용하여 파생 피처 DataFrame을 생성하는 핵심 함수 make_features를 구현하고 저장한 뒤, 첫 성공 실행 결과를 온전히 보존합니다.
  - 렉(Lag), 이동평균(MA), 계절지수 등 모든 파생 피처 산출 시 미래 수요를 절대 참조하지 않도록 엄격히 설계하며, 당월 운용시간은 사전에 알려진 계획값이라는 가정을 명시합니다.
- **AI 활용과 변경 과제**
  - AI와 기획 반영 및 변경 과제 수행 AI와의 협업을 통해 앞서 수립한 예측 모델 고도화 기획안을 실제 코드에 구체적으로 적용합니다.
  - 변경 과제: ma6, trend_t, lag2 피처를 제외하기 전후의 교차검증 결과를 상호 비교하고, 예측 시점에 실제로 알려진 계획 정보와 실측 정보를 명확히 구분하여 검증합니다.
  - 실제 코드와 실행 출력 결과물은 모두 안전하게 저장합니다.

#### 확인할 사항:

- 이 활동의 확인 근거 M04 구현: src/features/mnt02_feature_engineering.py의 make_features를저장하고계약〔hist DataFrame → 품목별 과거 수요와 제공된 계획 변수로 만든 파생 피처 DataFrame. lag/이동평균/계절지수는 미래 수요를 참조하지 않는다.
- 당월 운용시간은 사전 계획값이라는 가정을 명시한다.〕을충족하는 첫성공실행·로그·표/그림을보존했는가?
- 이후Lesson의추가변경을실제코드에반영했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 3: 근거 대조와 재실행

- **변경 코드 재실행**
  - 수정 코드 저장 및 재실행 검증 피처 제외 실험에 따른 수정 코드를 안전하게 저장하고 스크립트를 재실행합니다.
  - 피처 제외 실험 과정에서 동일한 확장창, 예측 기간, 오차 분모 조건을 엄격히 유지했는지 철저히 확인합니다.
- **AI 설명과 원자료 대조**
  - AI 설명 및 코드 선택 대조 분석 AI가 제시한 설명 및 코드 선택 사항 3건 이상을 실제 실습 자료 및 실행 결과 로그와 상호 대조하여 타당성을 엄격히 검토합니다.
  - 보완이 필요한 답변과 수정 사항을 다시 한번 확인하고, Prophet/LSTM이라는 원본 제목을 유지하되 실제 구현은 원본대로 포아송 HistGradientBoosting 모델로 수행되었음을 명확히 대조합니다.
  - 월별 한 달 앞 rolling 평가와 6개 월 고정 기점 예측의 차이를 명확히 구분하고, 당월 운용시간이 사전에 알려진 계획값이라는 가정 하에 수행되었는지 검증하여 실측값 입력으로 인한 정보 유출(Data Leakage)을 방지합니다.

#### 확인할 사항:

- **판정에 필요한 근거**
  - 이 활동의 확인 근거 M04 검증: [선택] 수요 예측 모델 — 파생 피처 10개 이상, 시계열 교차검증 평균 MAPE 25% 이하, 피처 중요도 근거와 결전 대비 수요 증가 품목을 제시했는가.
  - 실제기간·분모·조건·수치로판정하고 최초/수정실행을비교했는가?
- **미션별 확인 사항**
  - Prophet/LSTM이라는 원본 제목을 유지하되 실제 구현은 원본대로 포아송 HistGradientBoosting입니다.
  - 월별 한달 앞 rolling 평가와 6개 월 고정 기점 예측을 구분합니다.
  - 당월 운용시간은 사전에 알려진 계획값이라는 가정이 필요하고 실제 당월 실적이면 누수입니다.
- **AI 검토 기록**
  - AI 주장/선택3건을근거로검토·보완·재실행했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 4: 최종 판단과 산출물

- **산출물 제출과 상태 기록**
  - 종합 산출물 제출 및 상태 구분 원본 평가 기준에 따른 최종 결과물인 src/features/mnt02_feature_engineering.py, reports/mnt02_feature_validation.md, forms/MNT02_M04_forms.md의 실제 작성 내용, 최종 측정값, AI 대화 보존 위치, 선택 이유를 상호 연결하여 체계적으로 제출합니다.
  - 미달 항목, 미측정 항목, 문서 미작성 내용, 그리고 실제 승인 대기 상태를 명확히 구분하여 기록합니다.
- **미션별 검증 원칙**
  - 모델 구현 명시 및 정보 유출 방지 원본 제목인 Prophet/LSTM 명칭을 유지하되, 실제 구현은 환경 제약에 따라 포아송(Poisson) 기반의 HistGradientBoosting 모델로 수행되었음을 명확히 대조합니다.
  - 월별 한 달 앞 rolling 평가와 6개 월 고정 기점 예측의 차이를 엄격히 구분합니다.
  - 당월 운용시간은 사전에 알려진 계획값이라는 가정이 필요하며, 실제 당월 실적 데이터가 입력될 경우 정보 유출(Data Leakage)이 발생하므로 이를 철저히 방지했는지 최종 검증합니다.

#### 확인할 사항:

- 이 활동의 확인 근거 M04 완료: src/features/mnt02_feature_engineering.py, reports/mnt02_feature_validation.md, forms/MNT02_M04_forms.md의실제작성·최종값·대화보존위치·선택이유가연결되는가?
- 미달·미측정·문서미작성·실제승인대기를정확히표시했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

## 이 미션의 AI 활용 예시

- 가상 자료 분석·기획·코드 변경·디버깅·검증 설계·해석
- "교육용 미션 04의 핵심 함수 구현에서 누락되기 쉬운 입력값·경계조건을 알려주세요. 원본 자료에 없는 성능 수치를 만들지 말아 주세요."

AI 답변은 그대로 정답이 아닙니다. **실제 실행 로그와 최소 3건의 주장·선택을 비교**하세요.


## 원본 교관용 코드 — 참고 구현

다음은 제공된 Solution의 해당 미션 코드입니다. 실제 수행 시 환경에 필요한 패키지를 확인하고 실행하세요.


In [ ]:
from pathlib import Path

if not (globals().get('MNT02_PROJECT_READY', False) and globals().get('MNT02_PROJECT_DIR') and Path(MNT02_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    """과거 수요 지연값·이동평균·계절지수·훈련 계획·운용시간 등 파생 피처 15개로 시계열 머신러닝 수요 예측 모델(Prophet/LSTM 대신 포아송 그래디언트 부스팅)을 만들고, 시계열 교차검증으로 MAPE를 확인한 뒤 작전 템포 증가(결전 대비) 시 수요를 예측한다."""
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.ensemble import HistGradientBoostingRegressor
    from sklearn.inspection import permutation_importance

    # ---------------------------------------------------------------------------
    # [교육용가상자료] 가상 야전 정비창 수리부속 청구/출고·장비 가동률 데이터 생성기
    # 실제 부대·품목·재고·단가 정보와 무관하다. 품목 40개 x 48개월(가상 2022-01 ~ 2025-12) 월별 자료를 만든다.
    # ---------------------------------------------------------------------------
    FSC = {"2520": "power_train", "2530": "brake_steer", "2910": "fuel", "2930": "cooling",
           "2940": "filter", "5330": "gasket", "6140": "battery", "5961": "semiconductor"}
    EXERCISE = {3, 8, 10}      # 가상 대규모 훈련 월(훈련 계획으로 미리 알 수 있음)
    N_MONTHS = 48


    def season(fsc, month):
        """품목군별 계절 계수: 축전지는 겨울, 냉각계통은 여름에 수요가 늘어난다."""
        if fsc == "6140":
            return 1.6 if month in (12, 1, 2) else 0.9
        if fsc == "2930":
            return 1.5 if month in (6, 7, 8) else 0.9
        return 1 + 0.1 * np.sin(2 * np.pi * month / 12)


    def make_items(n=40, seed=2):
        """품목 마스터: NSN(가상), 품목군, 장비 유형, 기준 수요, 단가, 조달 리드타임, 공급업체 수, 생산(단종) 상태."""
        rng = np.random.default_rng(seed)
        rows = []
        for i in range(n):
            f = str(rng.choice(list(FSC)))
            status = str(rng.choice(["생산중", "단종예고", "단종"], p=[0.7, 0.2, 0.1]))
            rows.append({"item_id": f"P{i + 1:03d}", "nsn": f"{f}-00-{rng.integers(100, 1000)}-{rng.integers(1000, 10000)}",
                         "fsc": f, "group": FSC[f], "equip_type": str(rng.choice(["A", "B", "C"])),
                         "base": float(np.round(rng.lognormal(2.7, 0.8), 1)),
                         "price": int(rng.lognormal(12, 1.0)) // 100 * 100,
                         "lead_time": int(rng.integers(1, 7) if status == "생산중" else rng.integers(4, 13)),
                         "suppliers": int(rng.integers(1, 5)) if status == "생산중" else 1,
                         "status": status, "alt_available": bool(rng.random() < (0.6 if status == "생산중" else 0.3))})
        return pd.DataFrame(rows)


    def make_history(items, months=N_MONTHS, seed=3):
        """월별 청구 수요(개), 긴급 불출 건수, 장비 유형별 운용시간·가동률."""
        rng = np.random.default_rng(seed)
        t = np.arange(months)
        moy = t % 12 + 1
        ex = np.isin(moy, list(EXERCISE))
        hours = {e: (100 + 20 * rng.random(months)) * np.where(ex, 1.5, 1.0) for e in "ABC"}
        avail = {e: np.clip(0.93 - 0.12 * (hours[e] - 100) / 80 + rng.normal(0, 0.015, months), 0.6, 0.99) for e in "ABC"}
        rows = []
        for it in items.itertuples():
            trend = 1 + rng.uniform(-0.1, 0.3) * t / months
            s = np.array([season(it.fsc, m) for m in moy])
            lam = it.base * s * trend * hours[it.equip_type] / 110
            d = rng.poisson(lam)
            rows.append(pd.DataFrame({"item_id": it.item_id, "t": t, "ym": [f"{2022 + k // 12}-{k % 12 + 1:02d}" for k in t],
                                      "month": moy, "exercise": ex.astype(int), "op_hours": hours[it.equip_type].round(1),
                                      "availability": avail[it.equip_type].round(3), "demand": d,
                                      "emergency": rng.binomial(d, 0.05)}))
        return pd.concat(rows, ignore_index=True)


    def make_stock(items, hist, seed=4):
        """현재(가상 2026-01) 재고: 보유량과 미입고 발주량(조달 중인 물량). 단종 품목은 추가 발주가 어려워 미입고량이 적다."""
        rng = np.random.default_rng(seed)
        recent = hist[hist["t"] >= hist["t"].max() - 5].groupby("item_id")["demand"].mean()
        st = items[["item_id", "lead_time", "status"]].copy()
        st["on_hand"] = [int(round(recent[i] * rng.uniform(0.5, 3.0))) for i in st["item_id"]]
        cover = np.where(st["status"] == "생산중", rng.uniform(0.6, 1.3, len(st)), rng.uniform(0.0, 0.6, len(st)))
        st["on_order"] = [int(round(recent[i] * L * c)) for i, L, c in zip(st["item_id"], st["lead_time"], cover)]
        return st[["item_id", "on_hand", "on_order"]]
    # ---------------------------------------------------------------------------

    # ---- 수요 예측 공통 함수(파생 피처 + 머신러닝 모델) ----
    FEATS = ["lag1", "lag2", "lag3", "lag12", "ma3", "ma6", "std6", "season_idx", "month_sin", "month_cos",
             "exercise", "op_hours", "op_ratio", "item_level", "trend_t"]


    def make_features(hist):
        """품목별 과거 수요로 파생 피처를 만든다(모두 예측 시점 이전 정보만 사용)."""
        df = hist.sort_values(["item_id", "t"]).reset_index(drop=True)
        g = df.groupby("item_id")["demand"]
        for k in (1, 2, 3, 12):
            df[f"lag{k}"] = g.shift(k)
        df["ma3"] = g.transform(lambda s: s.shift(1).rolling(3).mean())
        df["ma6"] = g.transform(lambda s: s.shift(1).rolling(6).mean())
        df["std6"] = g.transform(lambda s: s.shift(1).rolling(6).std())
        df["item_level"] = g.transform(lambda s: s.shift(1).rolling(12).mean())
        df["season_idx"] = df["lag12"] / (g.transform(lambda s: s.shift(12).rolling(12).mean()) + 1)
        df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
        df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
        df["op_ratio"] = df["op_hours"] / df.groupby("item_id")["op_hours"].transform(lambda s: s.shift(1).rolling(12).mean())
        df["trend_t"] = df["t"]
        return df.dropna().reset_index(drop=True)


    def new_model():
        """Prophet/LSTM 대신 사용하는 포아송 손실 그래디언트 부스팅(개수 자료에 적합)."""
        return HistGradientBoostingRegressor(loss="poisson", max_iter=300, learning_rate=0.05, max_leaf_nodes=15,
                                             min_samples_leaf=20, random_state=0)


    def mape(y, p, min_y=1):
        y, p = np.asarray(y, float), np.asarray(p, float)
        m = y >= min_y
        return float(np.mean(np.abs(y[m] - p[m]) / y[m]))

    items = make_items()
    hist = make_history(items)
    df = make_features(hist)
    ab = set(items[items["base"] >= 10]["item_id"])                  # 수요가 충분한 품목(MAPE 평가 대상)
    print(f"파생 피처 {len(FEATS)}개 (목표 10개 이상): {', '.join(FEATS)}")

    # 1) 시계열 교차검증(확장 창): 과거로 학습 -> 바로 다음 6개월 평가, 3회 반복
    rows = []
    for cut in (30, 36, 42):
        tr, te = df[df["t"] < cut], df[(df["t"] >= cut) & (df["t"] < cut + 6)]
        p = new_model().fit(tr[FEATS], tr["demand"]).predict(te[FEATS])
        big = te["item_id"].isin(ab).to_numpy()
        rows.append({"학습 구간": f"t<{cut}", "평가 구간": f"t={cut}~{cut + 5}",
                     "ML MAE": round(np.abs(te["demand"] - p).mean(), 2), "MA3 MAE": round(np.abs(te["demand"] - te["ma3"]).mean(), 2),
                     "ML MAPE": mape(te["demand"][big], p[big]), "MA3 MAPE": mape(te["demand"][big], te["ma3"][big])})
    cv = pd.DataFrame(rows)
    print("\n[시계열 교차검증 결과 - MAPE는 수요 많은 품목 기준]")
    print(cv.assign(**{"ML MAPE": cv["ML MAPE"].map("{:.1%}".format), "MA3 MAPE": cv["MA3 MAPE"].map("{:.1%}".format)}).to_string(index=False))
    m_mape = cv["ML MAPE"].mean()
    print(f"평균 MAPE: ML {m_mape:.1%} / MA3 {cv['MA3 MAPE'].mean():.1%} -> 목표(MAPE 25% 이하): {'달성' if m_mape <= 0.25 else '미달'}")

    # 2) 최종 모델(t<42 학습)과 피처 중요도(순열 중요도)
    tr, te = df[df["t"] < 42], df[df["t"] >= 42]
    model = new_model().fit(tr[FEATS], tr["demand"])
    imp = permutation_importance(model, te[FEATS], te["demand"], n_repeats=5, random_state=0, scoring="neg_mean_absolute_error")
    imp = pd.Series(imp.importances_mean, index=FEATS).sort_values(ascending=False)
    print("\n[피처 중요도 상위 6개(MAE 증가량)]", ", ".join(f"{k} {v:.2f}" for k, v in imp.head(6).items()))
    print("중요도가 0에 가까운 피처(차원 축소 후보):", ", ".join(imp[imp < 0.02].index) or "없음")

    # 3) 결전 대비 시나리오: 다음 달(가상 2026-01) 운용시간 1.5배 + 훈련 월 지정
    future = []
    for it in items.itertuples():
        last = hist[(hist["item_id"] == it.item_id) & (hist["t"] == 47)].iloc[0]
        future.append({**last.to_dict(), "t": 48, "ym": "2026-01", "month": 1, "demand": 0, "exercise": 0})
    fut = pd.DataFrame(future)
    fut["op_hours"] = hist[hist["t"] == 36].set_index("item_id").loc[fut["item_id"], "op_hours"].to_numpy()  # 작년 1월 계획 운용시간
    res = {}
    for name, mult, ex in [("평시 계획", 1.0, 0), ("작전 템포 1.5배", 1.5, 1)]:
        f2 = fut.assign(op_hours=fut["op_hours"] * mult, exercise=ex)
        X = make_features(pd.concat([hist, f2], ignore_index=True))
        X = X[X["t"] == 48]
        res[name] = pd.Series(model.predict(X[FEATS]), index=X["item_id"])
    comp = pd.DataFrame(res).round(1)
    comp["증가율"] = (comp["작전 템포 1.5배"] / comp["평시 계획"] - 1).map("{:+.0%}".format)
    comp = comp.join(items.set_index("item_id")[["group", "status"]])
    print("\n[결전 대비 다음 달 수요 예측 - 증가량 상위 6개 품목]")
    print(comp.loc[(comp["작전 템포 1.5배"] - comp["평시 계획"]).sort_values(ascending=False).index[:6]].to_string())
    print(f"전체 예측 수요: 평시 {comp['평시 계획'].sum():.0f}개 -> 작전 템포 1.5배 {comp['작전 템포 1.5배'].sum():.0f}개")

    # 4) 예측 그림: 수요 많은 품목 1개
    iid = items.sort_values("base")["item_id"].iloc[-1]
    s = te[te["item_id"] == iid]
    full = df[df["item_id"] == iid]
    plt.figure(figsize=(9, 4))
    plt.plot(full["t"], full["demand"], "k.-", label="actual")
    plt.plot(s["t"], model.predict(s[FEATS]), "r-o", label="ML forecast")
    plt.plot(s["t"], s["ma3"], "b--", label="MA3 baseline")
    plt.axvline(41.5, c="gray", ls=":"); plt.xlabel("month index"); plt.ylabel("demand (EA)")
    plt.title(f"{iid} one-month-ahead forecast (synthetic)"); plt.legend(); plt.show()


## 원본 교관용 예상 출력 (참고용)

아래 내용은 원본에 포함된 예시이며 **실제 실행 결과로 제출할 수 없습니다.**

<details>
<summary><b>💡 [참고] 미션 04 예상 실행 결과 (Output)</b></summary>

```text
파생 피처 15개 (목표 10개 이상): lag1, lag2, lag3, lag12, ma3, ma6, std6, season_idx, month_sin, month_cos, exercise, op_hours, op_ratio, item_level, trend_t

[시계열 교차검증 결과 - MAPE는 수요 많은 품목 기준]
학습 구간   평가 구간  ML MAE  MA3 MAE ML MAPE MA3 MAPE
 t<30 t=30～35    5.85     7.26   25.8%    30.9%
 t<36 t=36～41    4.84     7.26   20.5%    30.9%
 t<42 t=42～47    4.57     7.34   20.8%    32.7%
평균 MAPE: ML 22.3% / MA3 31.5% -> 목표(MAPE 25% 이하): 달성

[피처 중요도 상위 6개(MAE 증가량)] lag12 7.83, item_level 7.20, op_hours 0.95, std6 0.27, ma3 0.19, op_ratio 0.13
중요도가 0에 가까운 피처(차원 축소 후보): ma6, trend_t, lag2

[결전 대비 다음 달 수요 예측 - 증가량 상위 6개 품목]
         평시 계획  작전 템포 1.5배   증가율        group status
item_id                                             
P016     138.9       168.2  +21%       gasket     단종
P028      74.1        99.5  +34%  power_train     단종
P033      73.3        94.1  +28%         fuel    생산중
P035      78.5        94.1  +20%  brake_steer    생산중
P030      43.4        56.8  +31%  brake_steer   단종예고
P026      15.6        24.3  +56%      cooling    생산중
전체 예측 수요: 평시 946개 -> 작전 템포 1.5배 1210개
```
</details>


## 교관 검토 기록

학생의 실제 실행 로그·대화 원문·수정 이력·평가 기준 충족 여부를 확인하고, 달성/미달/추가확인 및 승인 여부를 별도 기록합니다.

**통합 입력·저장 기능:** 마지막 통합 화면에서 미션별 산출물을 파일별로 작성·수정·저장하고, 실제 AI 질문·답변 원문, 원자료·출력 대조 근거, 검토 3건, 수정과 최종 수행 상태를 기록한 뒤 ZIP으로 다운로드합니다.

저장만으로 PASS나 승인이 되지 않습니다.


In [ ]:
# @title ⑤ 교관용 개별 산출물·검토기록 통합 입력, 저장 및 ZIP 다운로드 (실행 버튼을 누르세요) { display-mode: "form" }
# 입력 폼이 나타나면 소스코드 대신 버튼을 이용하세요. Colab/Jupyter에서 ipywidgets 필요.
from pathlib import Path

if not (globals().get('MNT02_PROJECT_READY', False) and globals().get('MNT02_PROJECT_DIR') and Path(MNT02_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    def _load_notebook_widgets():
        from importlib import import_module, invalidate_caches
        import subprocess
        import sys

        try:
            return import_module('ipywidgets')
        except ImportError:
            print('이 노트북 커널에 ipywidgets가 없어 설치합니다.')

        try:
            subprocess.run(
                [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                 '--retries', '0', '--timeout', '10', 'ipywidgets'],
                check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=60,
            )
            invalidate_caches()
            return import_module('ipywidgets')
        except (subprocess.SubprocessError, OSError, ImportError):
            print('이 커널에서 ipywidgets를 설치하거나 불러오지 못했습니다.')
            print('설치 가능한 환경에서는 새 셀에서 %pip install ipywidgets 를 실행한 뒤 이 셀을 다시 실행하세요.')
            print('설치할 수 없는 환경에서는 제공된 Markdown 작성 양식을 사용하세요.')
            return None

    widgets = _load_notebook_widgets()
    if widgets is not None:
        import json, csv, io, ast, zipfile, datetime, pathlib
        from pathlib import Path
        import os
        from urllib.parse import quote
        from IPython.display import HTML, display, Markdown, clear_output

        MISSION_CODE = 4
        DELIVERABLE_PATHS = ['forms/MNT02_M04_forms.md', 'reports/mnt02_feature_validation.md', 'src/features/mnt02_feature_engineering.py']
        PROJECT_NAME = 'MNT02'
        NOTEBOOK_ROLE = '교관용 Solution'
        WORK_ROOT = Path(MNT02_PROJECT_DIR).resolve()

        RECORD_DIR = WORK_ROOT / f'MNT02_M{MISSION_CODE:02d}_학습기록'
        RECORD_DIR.mkdir(parents=True, exist_ok=True)
        ZIP_FILENAME = f'MNT02_m{MISSION_CODE:02d}_review.zip'

        # 반드시 CMS 지정 경로만 사용하며 상대 경로의 작업공간 이탈을 방지합니다.
        def safe_path(relative):
            target = (WORK_ROOT / relative).resolve()
            if not target.is_relative_to(WORK_ROOT):
                raise ValueError('작업공간 밖의 경로는 사용할 수 없습니다.')
            return target

        def valid_text(path, content):
            suffix = Path(path).suffix.lower()
            if suffix == '.json':
                json.loads(content)
            elif suffix == '.ipynb':
                doc = json.loads(content)
                if not isinstance(doc, dict) or not isinstance(doc.get('cells'), list) or 'nbformat' not in doc:
                    raise ValueError('노트북 JSON에 cells 또는 nbformat 정보가 없습니다.')
            elif suffix == '.py':
                ast.parse(content)
            elif suffix == '.csv':
                if not content.strip():
                    raise ValueError('CSV 데이터가 비어 있습니다.')
                rows = list(csv.reader(io.StringIO(content)))
                if not rows or not rows[0] or not any(x.strip() for x in rows[0]):
                    raise ValueError('CSV 첫 행에 열 이름을 입력하세요.')
                if any(len(row) != len(rows[0]) for row in rows if row):
                    raise ValueError('CSV 열 개수가 행마다 다릅니다.')

        # 산출물 입력 화면 — 파일별 별도 탭
        editors = {}
        file_tabs = []
        file_buttons = {}
        for relative in DELIVERABLE_PATHS:
            suffix = Path(relative).suffix.lower()
            hint = {
                '.md': '보고서·양식의 제목, 분석 결과, 근거와 수정 이유를 Markdown으로 작성합니다.',
                '.csv': '첫 줄에 열 이름, 다음 줄부터 데이터 행을 쉼표로 구분하여 입력합니다.',
                '.json': '키와 값이 있는 유효한 JSON 형식으로 실제 결과를 입력합니다.',
                '.py': '직접 구현하고 확인한 Python 소스코드를 입력합니다.',
                '.ipynb': '노트북 JSON 원문을 입력하거나 기존 파일에서 불러옵니다. JSON 노트북 형식을 유지하세요.'
            }.get(suffix, '실제 작성한 내용을 입력합니다.')
            editor = widgets.Textarea(value='', placeholder='여기에 내용을 작성하세요. 입력하지 않은 파일은 저장하지 않습니다.',
                                      layout=widgets.Layout(width='100%', height='250px'))
            status = widgets.HTML('<span style="color:#777">아직 저장하지 않았습니다.</span>')
            load_btn = widgets.Button(description='기존 파일 불러오기', icon='folder-open', button_style='')
            save_btn = widgets.Button(description='이 파일 저장', icon='save', button_style='success')
            editors[relative] = {'editor': editor, 'status': status, 'loaded': False, 'saved': False, 'dirty': False, 'saved_value': ''}
            def on_edit(change, key=relative):
                state = editors[key]
                state['dirty'] = change['new'] != state['saved_value']
                state['saved'] = state['loaded'] and not state['dirty']
                if state['dirty']:
                    state['status'].value = '<span style="color:#a65">● 수정됨 — 저장 버튼을 누르세요.</span>'
                elif state['saved']:
                    state['status'].value = '<span style="color:#286">저장된 내용과 같습니다.</span>'
                else:
                    state['status'].value = '<span style="color:#777">아직 저장하지 않았습니다.</span>'
            editor.observe(on_edit, names='value')
            def load_clicked(button, key=relative):
                state = editors[key]
                target = safe_path(key)
                if not target.is_file():
                    state['status'].value = '<span style="color:#a22">파일이 아직 없습니다. 직접 입력 후 저장하세요.</span>'
                    return
                if state['dirty']:
                    state['status'].value = '<span style="color:#a22">미저장 내용이 있습니다. 불러오기 전에 저장하세요.</span>'
                    return
                try:
                    state['loaded'] = False
                    state['saved_value'] = target.read_text(encoding='utf-8')
                    state['editor'].value = state['saved_value']
                    state['loaded'] = True
                    state['dirty'] = False
                    state['saved'] = True
                    state['status'].value = '<span style="color:#286">파일을 불러왔습니다. 변경하면 다시 저장하세요.</span>'
                except Exception as exc:
                    state['status'].value = '<span style="color:#a22">불러오기 오류: '+str(exc)+'</span>'
            def save_clicked(button, key=relative):
                save_one(key)
            load_btn.on_click(load_clicked)
            save_btn.on_click(save_clicked)
            file_buttons[relative] = {'load':load_btn,'save':save_btn}
            file_tabs.append(widgets.VBox([
                widgets.HTML('<b>저장 경로:</b> <code>'+relative+'</code><br><span>'+hint+'</span><br><span style="color:#a55">기존 파일이 있다면 저장 시 덮어씁니다. 실습 완료·승인과는 다릅니다.</span>'),
                editor, widgets.HBox([load_btn, save_btn]), status
            ]))

        def save_one(relative):
            state = editors[relative]
            value = state['editor'].value
            if not value.strip():
                state['status'].value = '<span style="color:#a22">비어 있어 저장하지 않았습니다.</span>'
                return False
            try:
                valid_text(relative, value)
                target = safe_path(relative)
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_text(value, encoding='utf-8')
                state['saved_value'] = value
                (WORK_ROOT / ZIP_FILENAME).unlink(missing_ok=True)
                state['loaded'] = True
                state['dirty'] = False
                state['saved'] = True
                state['status'].value = '<span style="color:green">✓ 저장 완료: '+relative+'</span>'
                return True
            except Exception as exc:
                state['status'].value = '<span style="color:#a22">저장 실패: '+str(exc)+'</span>'
                return False

        # 학습기록 및 AI 원문, 자료 대조, 검토 3건 이상
        record_specs = [
         ('plan','실습 계획·검토 기록','이번 미션에서 직접 수행한 내용, 실행 조건 및 최초 결과를 작성하세요.'),
         ('ai_request','실제 AI 질문 원문','실제로 입력한 질문 원문만 작성하세요. 사용하지 않았다면 비워 둡니다.'),
         ('ai_response','실제 받은 AI 답변 원문','실제로 받은 답변 전체를 붙여 넣으세요. 임의로 생성한 대화는 기록하지 않습니다.'),
         ('evidence','실제 출력·자료 대조 근거','실제 출력값, 원자료 위치, 전후 비교 및 계산 근거를 적으세요.'),
         ('revision','검토 및 수정 내용 종합','검토 3건의 발견사항과 코드를 수정·재실행한 기록을 종합하세요.'),
         ('decision','최종 판단과 미달·미측정·미실시 사유','달성 여부와 남은 부분, 근거 및 추가 확인사항을 적으세요.'),
        ]
        record_widgets = {}
        record_items = []
        for key, title, hint in record_specs:
            field = widgets.Textarea(placeholder=hint, layout=widgets.Layout(width='100%',height='115px'))
            record_widgets[key] = field
            record_items.append(widgets.VBox([widgets.HTML('<b>'+title+'</b><br><small>'+hint+'</small>'),field]))

        reviews = []
        for i in range(3):
            claim = widgets.Textarea(placeholder=f'{i+1}번째 AI 주장·코드 선택 또는 점검 대상',layout=widgets.Layout(width='100%',height='65px'))
            ground = widgets.Textarea(placeholder='원자료·실행 출력의 위치 및 비교한 근거',layout=widgets.Layout(width='100%',height='65px'))
            action = widgets.Textarea(placeholder='맞음·수정·보류, 수정 내용 및 재확인 결과',layout=widgets.Layout(width='100%',height='65px'))
            reviews.append((claim,ground,action))
            record_items.append(widgets.VBox([widgets.HTML(f'<b>검토 {i+1} (최소 3건 중 {i+1}번째)</b>'),claim,ground,action]))

        status_check = widgets.SelectMultiple(options=['달성','미달','미측정','미실시','추가 확인 필요'],
                                              value=(),description='수행 상태',layout=widgets.Layout(width='70%',height='115px'))
        approval = widgets.Dropdown(options=['미확인 / 교관 승인 대기','실제 교관 검토 완료 (기록 확인 필요)'],
                                    value='미확인 / 교관 승인 대기', description='교관 상태',layout=widgets.Layout(width='85%'))
        record_notice = widgets.HTML('<span style="color:#666">입력 후 [학습기록 저장]을 누르세요. 미작성·미측정은 그대로 표시합니다.</span>')
        record_dirty = {'value': False}
        def _mark_record_dirty(change):
            record_dirty['value'] = True
            record_notice.value = '<span style="color:#a65">● 학습기록에 저장하지 않은 변경이 있습니다.</span>'
        for field in list(record_widgets.values()) + [v for group in reviews for v in group] + [status_check,approval]:
            field.observe(_mark_record_dirty,names='value')

        # 앞서 작성한 학습기록을 자동으로 불러와 이어서 수정할 수 있습니다.
        previous_audit = RECORD_DIR/'learner_audit.json'
        if previous_audit.is_file():
            try:
                previous = json.loads(previous_audit.read_text(encoding='utf-8'))
                for key, field in record_widgets.items():
                    field.value = str(previous.get(key,'') or '')
                for group, past in zip(reviews, previous.get('reviews',[])):
                    for widget, key in zip(group,('claim','evidence','revision_and_recheck')):
                        widget.value = str(past.get(key,'') or '')
                status_check.value = tuple(x for x in previous.get('status',[]) if x in status_check.options)
                if previous.get('instructor_approval') in approval.options:
                    approval.value = previous['instructor_approval']
                record_dirty['value'] = False
                record_notice.value = '<span style="color:#286">이전 학습기록을 불러왔습니다. 수정 후 다시 저장하세요.</span>'
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">기존 학습기록 읽기 오류: '+str(exc)+'</span>'

        def audit_object():
            return {
                'project': PROJECT_NAME, 'mission': MISSION_CODE, 'notebook_role': NOTEBOOK_ROLE,
                'plan':record_widgets['plan'].value, 'ai_request':record_widgets['ai_request'].value,
                'ai_response':record_widgets['ai_response'].value, 'evidence':record_widgets['evidence'].value,
                'revision':record_widgets['revision'].value, 'decision':record_widgets['decision'].value,
                'reviews':[{'claim':a.value,'evidence':b.value,'revision_and_recheck':c.value} for a,b,c in reviews],
                'status':list(status_check.value),'instructor_approval':approval.value,
                'saved_at':datetime.datetime.now().isoformat(timespec='seconds')
            }

        def save_audit(button=None):
            try:
                record = audit_object()
                (RECORD_DIR/'learner_audit.json').write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding='utf-8')
                (RECORD_DIR/'AI_대화원문.txt').write_text('실제 AI 질문 원문\n'+record['ai_request']+'\n\n실제 AI 답변 원문\n'+record['ai_response'],encoding='utf-8')
                sections = [f'# MNT02 미션 {MISSION_CODE:02d} 학습기록']
                for name,key in [('실습 계획·검토 기록','plan'),('실제 출력·자료 대조 근거','evidence'),('검토 및 수정 내용','revision'),('최종 판단','decision')]:
                    sections.append('## '+name+'\n'+(record[key] or '(미작성)'))
                for i,item in enumerate(record['reviews'],1):
                    sections.append(f"## 검토 {i}\n- 검토 대상: {item['claim'] or '(미작성)'}\n- 대조 근거: {item['evidence'] or '(미작성)'}\n- 수정 및 재확인: {item['revision_and_recheck'] or '(미작성)'}")
                sections.append('## 상태\n'+(', '.join(record['status']) or '미선택')+'\n\n교관 상태: '+record['instructor_approval'])
                (RECORD_DIR/'학습기록.md').write_text('\n\n'.join(sections)+'\n',encoding='utf-8')
                record_dirty['value'] = False
                (WORK_ROOT / ZIP_FILENAME).unlink(missing_ok=True)
                completed = sum(all(v.strip() for v in [a.value,b.value,c.value]) for a,b,c in reviews)
                record_notice.value = f'<span style="color:green">✓ 학습기록 저장 완료. 근거가 모두 입력된 검토: {completed}/3건. 저장은 PASS나 교관 승인과 다릅니다.</span>'
                return True
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">학습기록 저장 실패: '+str(exc)+'</span>'
                return False

        save_audit_btn = widgets.Button(description='학습기록 저장',icon='save',button_style='success')
        save_audit_btn.on_click(save_audit)
        record_tab = widgets.VBox(record_items+[widgets.HTML('<b>최종 수행 상태</b> — 해당되는 항목을 선택하세요. 선택만으로 자동 평가되지 않습니다.'),status_check,approval,save_audit_btn,record_notice])

        tabs = widgets.Tab(children=file_tabs+[record_tab])
        for i, path in enumerate(DELIVERABLE_PATHS):
            tabs.set_title(i, f'산출물 {i+1}')
        tabs.set_title(len(file_tabs),'학습기록')

        summary_output = widgets.Output()
        all_btn = widgets.Button(description='작성한 산출물 전체 저장', icon='save',button_style='info')
        zip_btn = widgets.Button(description='저장 파일 ZIP 생성',icon='archive',button_style='warning')
        dl_btn = widgets.Button(description='ZIP 다운로드',icon='download',button_style='')

        def save_all(button):
            ok = 0
            for rel in DELIVERABLE_PATHS:
                if editors[rel]['editor'].value.strip() and save_one(rel):
                    ok += 1
            with summary_output:
                summary_output.clear_output()
                print(f'산출물 {ok}개 저장 완료. 파일별 상태를 확인하세요. 학습기록은 별도 [학습기록 저장] 버튼을 누르세요.')

        def build_zip(button):
            with summary_output:
                summary_output.clear_output()
                if record_dirty['value']:
                    print('❌ 학습기록에 미저장 변경 내용이 있습니다. [학습기록 저장]을 먼저 누르세요.')
                    return
                dirty = [p for p,s in editors.items() if s['dirty']]
                if dirty:
                    print('❌ 미저장 변경 내용이 있습니다. 아래 파일들을 먼저 저장하세요:')
                    for p in dirty: print(' -',p)
                    return
                # 기존에 제공된 파일은 ZIP에 임의로 포함하지 않고 현재 화면에서 불러오거나 저장한 파일만 수집합니다.
                saved = [p for p,s in editors.items() if s['saved'] and safe_path(p).is_file()]
                audit_files = [p for p in RECORD_DIR.rglob('*') if p.is_file()]
                if not saved and not audit_files:
                    print('❌ 저장된 결과물이 없습니다. 산출물 또는 학습기록을 저장하세요.')
                    return
                dest = WORK_ROOT / ZIP_FILENAME
                with zipfile.ZipFile(dest,'w',compression=zipfile.ZIP_DEFLATED) as archive:
                    for rel in saved: archive.write(safe_path(rel), arcname=rel)
                    for f in audit_files: archive.write(f, arcname=str(f.relative_to(WORK_ROOT)))
                print('✅ ZIP 생성:', dest)
                print('산출물:',len(saved),'개 / 학습기록:',len(audit_files),'개')
                print('※ ZIP 생성은 학습 완료 또는 교관 승인을 의미하지 않습니다.')

        def download_zip(button):
            with summary_output:
                summary_output.clear_output()
                target = WORK_ROOT / ZIP_FILENAME
                if record_dirty['value'] or any(v['dirty'] for v in editors.values()):
                    print('미저장 변경이 있습니다. 먼저 저장 후 ZIP을 다시 만드세요.')
                    return
                if not target.is_file():
                    print('먼저 [저장 파일 ZIP 생성]을 눌러 주세요.')
                    return
                try:
                    from google.colab import files as colab_files
                    colab_files.download(str(target))
                    print('다운로드를 요청했습니다.')
                except ImportError:
                    start_dir = globals().get('notebook_directory', WORK_ROOT.parent)
                    href = quote(Path(os.path.relpath(target, start_dir)).as_posix())
                    display(HTML(f'<a href="{href}" download>ZIP 다운로드: {target.name}</a>'))

        all_btn.on_click(save_all)
        zip_btn.on_click(build_zip)
        dl_btn.on_click(download_zip)

        print(f'✅ MNT02 미션 {MISSION_CODE:02d} ({NOTEBOOK_ROLE}) 산출물·학습기록 입력 화면')
        display(Markdown('**사용 순서:** ① 각 산출물 탭에서 작성하거나 기존 파일 불러오기 → ② 파일별 저장 → ③ 학습기록 탭에서 실제 AI 원문·근거·검토 3건·최종 상태 입력 및 저장 → ④ ZIP 생성 → ⑤ 다운로드.'))
        display(Markdown('**주의:** 교관용 해설 및 예상 출력은 실제 실행·승인 증거가 아닙니다. 입력창은 실제 수행 내용을 기록하기 위한 기능이며, 미작성 결과를 자동으로 완성하지 않습니다.'))
        display(tabs,widgets.HBox([all_btn,zip_btn,dl_btn]),summary_output)
